# Warm-up 1A · VN30 Next-Day Return · Dự báo lợi suất ngày mai (Regression · Hồi quy)

| | 🇬🇧 English | 🇻🇳 Tiếng Việt |
|---|---|---|
| **Task** | Predict `next_ret`, tomorrow's return (%) of each stock, from indicators at today's close. | Dự báo `next_ret` — lợi suất ngày mai (%) của từng cổ phiếu — từ các chỉ báo lúc đóng cửa hôm nay. |
| **Data** | `train/` 2021–2024 (with label) · `public_test/` 2025 H1 · `private_test/` 2025 H2 | `train/` 2021–2024 (có nhãn) · `public_test/` 6T đầu 2025 · `private_test/` 6T cuối 2025 |
| **Metric** | RMSE, **lower is better**. Ranking uses the **private** score. | RMSE, **càng thấp càng tốt**. Xếp hạng theo điểm **private**. |
| **Submission** | `public_submission.csv`, `private_submission.csv` with columns `id,next_ret` | `public_submission.csv`, `private_submission.csv` gồm cột `id,next_ret` |
| **Rules** | PyTorch model, no external data. | Mô hình PyTorch, không dùng dữ liệu ngoài. |

$$\text{RMSE}=\sqrt{\tfrac{1}{N}\textstyle\sum_{i}(y_i-\hat y_i)^2}$$

**Columns · Các cột:** `id, date, ticker, sector, ret_1d, ret_5d, ret_20d, volatility_20d, rsi_14, volume_ratio,
foreign_net_buy_bn` (has NaN · có giá trị thiếu)`, market_ret_1d, usd_vnd_change, day_of_week, next_ret` (target · nhãn).

🇬🇧 This baseline runs end-to-end and submits a result. Beating it is up to you.
🇻🇳 Baseline này chạy trọn vẹn và nộp được kết quả. Vượt qua nó là việc của bạn.

## 1. Setup and Imports · Cài đặt và thư viện

In [ ]:
import os
import re
import shutil
import unicodedata
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration and Hyperparameters · Cấu hình và siêu tham số
🇬🇧 Nothing to fill in here: the data downloads itself, and your team comes from your account on the site when you submit (section 7).
🇻🇳 Không cần điền gì ở đây: dữ liệu tự tải về, và đội của bạn được lấy từ tài khoản trên trang web khi nộp bài (phần 7).

In [ ]:
# ---- Data / Dữ liệu ----
# The dataset downloads itself from GitHub straight onto this Colab session's own
# disk. No Google Drive and no mounting: reading thousands of small files over a
# Drive mount is what made the old flow slow, and local disk is several times
# faster per epoch.
# Bộ dữ liệu tự tải từ GitHub vào ổ đĩa của phiên Colab này. Không cần Drive.
TASK    = "1A"
FOLDER  = "1A_stock_regression"
RELEASE = "https://github.com/ftulabs/olympiad-datasets/releases/download/warmup-1"

# Already have the folder? Point this at the parent of 1A_stock_regression/ and nothing is
# downloaded. / Đã có sẵn thư mục? Trỏ vào thư mục cha của 1A_stock_regression/.
ROOT_OVERRIDE = ""

import os, subprocess, time

ROOT = ROOT_OVERRIDE or "/content/olympiad"
if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    os.makedirs(ROOT, exist_ok=True)
    zip_path = f"{ROOT}/{FOLDER}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {FOLDER}.zip / Đang tải ...", flush=True)
        t0 = time.time()
        subprocess.run(["curl", "-fsSL", "--retry", "3", "-o", zip_path + ".part",
                        f"{RELEASE}/{FOLDER}.zip"], check=True)
        os.replace(zip_path + ".part", zip_path)  # only a complete download gets the final name
        print(f"  {os.path.getsize(zip_path)/1e6:.0f} MB in {time.time()-t0:.0f}s", flush=True)
    print("Extracting / Đang giải nén ...", flush=True)
    subprocess.run(["unzip", "-nq", zip_path, "-d", ROOT], check=True)

if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    raise FileNotFoundError(
        f"{ROOT}/{FOLDER}/dataset is missing after the download. Re-run this cell. "
        f"/ Thiếu dữ liệu sau khi tải. Hãy chạy lại ô này."
    )
print(f"Data root / Thư mục dữ liệu: {ROOT}")

DATA_DIR = f"{ROOT}/1A_stock_regression/dataset"
TRAIN_CSV = f"{DATA_DIR}/train/train.csv"
PUBLIC_CSV = f"{DATA_DIR}/public_test/public_test.csv"
PRIVATE_CSV = f"{DATA_DIR}/private_test/private_test.csv"
SAVE_DIR = "./checkpoints"

# ---- Hyperparameters / Siêu tham số ----
BATCH_SIZE = 256
EPOCHS = 10
LR = 1e-3
HIDDEN = 64
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs(SAVE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3. Data Loading and Preprocessing · Đọc và tiền xử lý dữ liệu

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
public_df = pd.read_csv(PUBLIC_CSV)
private_df = pd.read_csv(PRIVATE_CSV)
print(f"Train: {len(train_df)} | Public test: {len(public_df)} | Private test: {len(private_df)}")

TARGET = "next_ret"
FEATURES = ["ret_1d", "ret_5d", "ret_20d", "volatility_20d", "rsi_14", "volume_ratio",
            "foreign_net_buy_bn", "market_ret_1d", "usd_vnd_change", "day_of_week"]

# Validation = last 20% of trading days (the test set is in the future)
# Tập validation = 20% số phiên cuối cùng (vì tập test nằm ở tương lai)
dates = np.sort(train_df["date"].unique())
is_val = train_df["date"] >= dates[int(0.8 * len(dates))]

mean = train_df.loc[~is_val, FEATURES].mean()
std = train_df.loc[~is_val, FEATURES].std()

def to_tensor(df):
    x = ((df[FEATURES] - mean) / std).fillna(0.0)
    return torch.tensor(x.values, dtype=torch.float32)

X_train, y_train = to_tensor(train_df[~is_val]), torch.tensor(train_df.loc[~is_val, TARGET].values, dtype=torch.float32)
X_val, y_val = to_tensor(train_df[is_val]), torch.tensor(train_df.loc[is_val, TARGET].values, dtype=torch.float32)
X_public, X_private = to_tensor(public_df), to_tensor(private_df)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
print(f"Train: {len(X_train)} | Validation: {len(X_val)} | Features: {X_train.shape[1]}")
train_df.head()

## 4. Model Architecture · Kiến trúc mô hình

In [ ]:
class MLP(nn.Module):
    """Small fully-connected network. / Mạng nơ-ron truyền thẳng nhỏ."""

    def __init__(self, n_in, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


model = MLP(X_train.shape[1]).to(DEVICE)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training · Huấn luyện

In [ ]:
@torch.no_grad()
def predict(model, X):
    model.eval()
    return model(X.to(DEVICE)).cpu().numpy()


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - y_pred) ** 2)))


criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_rmse = float("inf")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)

    val_rmse = rmse(y_val, predict(model, X_val))
    msg = f"Epoch {epoch:02d} | Loss={total_loss / len(X_train):.4f} | Val RMSE={val_rmse:.4f}"
    if val_rmse < best_rmse:
        best_rmse = val_rmse
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pt"))
        msg += "  ✓ best model saved"
    print(msg)

print(f"\nBest validation RMSE: {best_rmse:.4f}")

## 6. Predict Public & Private Test · Dự đoán tập Public & Private
🇬🇧 Creates `public_submission.csv` / `private_submission.csv` (+ `.zip`), like the official competition.
🇻🇳 Tạo `public_submission.csv` / `private_submission.csv` (+ `.zip`), giống định dạng cuộc thi chính thức.

In [ ]:
def export(ids, preds, csv_path):
    """Save predictions as <id, TARGET> CSV and zip it. / Lưu dự đoán ra CSV và nén zip."""
    pd.DataFrame({"id": ids, TARGET: preds}).to_csv(csv_path, index=False)
    zip_path = csv_path.replace(".csv", ".zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=os.path.basename(csv_path))
    print(f"Saved {csv_path} and {zip_path}")


model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best_model.pt"), map_location=DEVICE))
export(public_df["id"], predict(model, X_public), "public_submission.csv")
export(private_df["id"], predict(model, X_private), "private_submission.csv")
pd.read_csv("public_submission.csv").head()

## 7. Submit · Nộp bài

🇬🇧 Your team comes from your account on [seleccion.ftds.online](https://seleccion.ftds.online/), not from a name typed here. The cell below prints a short code once per Colab session; type it on the site and this notebook submits as you.

🇻🇳 Đội của bạn lấy từ tài khoản trên [seleccion.ftds.online](https://seleccion.ftds.online/). Ô bên dưới in ra một mã ngắn mỗi phiên Colab; nhập mã đó trên trang web.

In [ ]:
def check_submission(csv_path, test_df):
    """Validate the file format before submitting. / Kiểm tra định dạng file trước khi nộp."""
    sub = pd.read_csv(csv_path)
    assert list(sub.columns) == ["id", TARGET], f"{csv_path}: columns must be ['id', '{TARGET}']"
    assert len(sub) == len(test_df) and set(sub["id"]) == set(test_df["id"]), f"{csv_path}: ids do not match the test set"
    assert sub[TARGET].notna().all(), f"{csv_path}: contains missing values / có giá trị bị thiếu"


check_submission("public_submission.csv", public_df)
check_submission("private_submission.csv", private_df)

In [ ]:
# ===================================================================
#  Olympiad AI - submit.  Paste at the END of your notebook, then run.
#  No key to copy: the first run prints a code you enter on the site.
# ===================================================================
BASE    = "https://seleccion.ftds.online"
PROBLEM = "1A"          # which problem this notebook is for
CONTEST = ""            # blank = the contest you are entered in
PUBLIC  = "public_submission.csv"
PRIVATE = "private_submission.csv"
# -------------------------------------------------------------------
import os, time, requests

token = globals().get("_OLY_TOKEN")
if not token:
    start = requests.post(f"{BASE}/api/pair/start", timeout=30).json()
    print(f"\n  1. open   {start['enter_at']}")
    print(f"  2. enter  {start['code']}\n  waiting...", flush=True)
    for _ in range(int(start["expires_in"]) // 3):
        time.sleep(3)
        got = requests.get(f"{BASE}/api/pair/poll",
                           params={"code": start["code"]}, timeout=30).json()
        if got.get("ready"):
            token = globals()["_OLY_TOKEN"] = got["token"]
            print("  paired ✓ (this notebook stays paired for the session)")
            break
    else:
        raise SystemExit("pairing timed out - run this cell again")

missing = [f for f in (PUBLIC, PRIVATE) if not os.path.exists(f)]
if PUBLIC in missing:
    raise SystemExit(f"{PUBLIC} not found - run the cell that writes it first")

files = {"public": open(PUBLIC, "rb")}
if PRIVATE not in missing:
    files["private"] = open(PRIVATE, "rb")
try:
    r = requests.post(f"{BASE}/api/submit", headers={"X-Oly-Token": token},
                      data={"problem": PROBLEM, "contest": CONTEST},
                      files=files, timeout=600)
finally:
    for f in files.values():
        f.close()

try:
    d = r.json()
except ValueError:
    raise SystemExit(f"server {r.status_code}: {r.text[:300]}")

if r.status_code == 429:
    print(f"⏳ cooldown - wait {d.get('retry_after')}s and run again")
elif r.status_code == 401:
    globals().pop("_OLY_TOKEN", None)
    print(f"❌ {d.get('detail')}  (run this cell again to re-pair)")
elif r.status_code != 200:
    print(f"❌ {r.status_code}: {d.get('detail') or d.get('message')}")
elif d["public"]["status"] != "ok":
    # the grader's own message about your file - it says what to fix
    print(f"❌ {d['public']['status'].upper()}: {d['public']['message']}")
else:
    p, c = d["public"], (d.get("counting") or {})
    print(f"✅ {d['problem']} · {d['title']}")
    print(f"   {d['metric']} = {p['score']:.4f}   {p['tier'] or '(below the lowest tier)'}")
    print(f"   entry #{d['entries']}   ·   rank: {d['rank']}")
    if c.get("score") is not None and abs(c["score"] - p["score"]) > 1e-12:
        print(f"   counting entry: {c['score']:.4f} {c['tier']}")
    print(f"   {BASE}/")